# Uji sensitivitas: Forward-Forward pada learning rate 10⁻³
**Tujuan.** Di eksperimen utama, FF memilih LR terbesar di grid (3·10⁻³). Reviewer bisa berargumen bahwa hilangnya plastisitas FF disebabkan step size yang besar, bukan lokalitas. Notebook ini menjalankan FF dengan **LR 10⁻³**, dengan konfigurasi lain **identik** (dibaca dari `results_v2/config.json`), 5 seed × 2 stream.

**Interpretasi.** Jika retention FF @ 10⁻³ tetap jauh di bawah BP-G (≈ 0,99) → kesimpulan "lokalitas tidak melindungi plastisitas" diperkuat. Jika naik mendekati BP-G → step size berperan, dan klaim di naskah disesuaikan.

**Cara pakai.** Runtime → T4 GPU → **Run all**. Perkiraan ±25–30 menit. Tahan putus: run dilanjutkan dari checkpoint di Drive; run yang selesai dilewati. Hasil: `results_v2_lrsens/`, tabel `figures_v2/lr_sensitivity.csv`, gambar `figures_v2/paper/Figure_A1`, dan satu ZIP yang otomatis diunduh.

## 1. Drive + GPU

In [ ]:
import os, sys, json, time
SESSION_START = time.time()
import torch
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ff_plasticity'
RES_MAIN = f'{BASE}/results_v2'
RES_S = f'{BASE}/results_v2_lrsens'
FIG = f'{BASE}/figures_v2'
for d in (RES_S, FIG, f'{FIG}/paper'): os.makedirs(d, exist_ok=True)
def assert_drive():
    if BASE.startswith('/content/drive') and not os.path.ismount('/content/drive'):
        raise RuntimeError('Google Drive terlepas! Jalankan ulang sel 1.')
assert_drive()
assert os.path.exists(f'{RES_MAIN}/config.json'), 'results_v2/config.json tidak ditemukan — jalankan notebook utama v2/v3 dulu.'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 2. Pustaka + konfigurasi yang sama persis dengan eksperimen utama

In [ ]:
sys.path.insert(0, BASE)
import importlib, ffplast as fp
importlib.reload(fp)
CFG = json.load(open(f'{RES_MAIN}/config.json'))
BENCH, WIDTH, DEPTH = CFG['BENCH'], CFG['WIDTH'], CFG['DEPTH']
KW = {k: CFG[k] for k in ('optimizer', 'theta', 'label_energy', 'rho', 'maturity')}
LR_MAIN = json.load(open(f'{RES_MAIN}/lr_choice.json'))
LR_SENS = 1e-3
SEEDS = [0, 1, 2, 3, 4]
METHODS = ['ff']                  # tambahkan 'ff_cbp' bila ingin menguji FF + reset juga
BUDGET_MIN = 60
DEADLINE = SESSION_START + BUDGET_MIN * 60
print('BENCH:', {k: (v['n_train'], v['epochs'], v['n_tasks']) for k, v in BENCH.items()})
print('LR utama FF:', {b: LR_MAIN[b]['ff'] for b in BENCH}, '-> uji:', LR_SENS)

## 3. Data (dari cache Drive)

In [ ]:
DATA = {}
for bk, cfg in BENCH.items():
    DATA[bk] = fp.load_dataset(cfg['dataset'], f'{BASE}/data')
    print(bk, tuple(DATA[bk][0].shape))

## 4. Jalankan FF @ LR 10⁻³ (dapat dilanjutkan otomatis)

In [ ]:
for seed in SEEDS:
    for bk, cfg in BENCH.items():
        for m in METHODS:
            if time.time() > DEADLINE:
                break
            assert_drive()
            try:
                fp.run(cfg, m, seed, WIDTH, DEPTH, LR_SENS, RES_S, DATA[bk], DEVICE,
                       ckpt_every=10, deadline=DEADLINE, **KW)
            except Exception as e:
                print('ERROR', bk, m, seed, repr(e))
    print(f'== seed {seed} selesai | {(time.time() - SESSION_START) / 60:.1f} menit ==')
done = [f for f in os.listdir(RES_S) if f.endswith('.done')]
print(len(done), 'dari', len(SEEDS) * len(BENCH) * len(METHODS), 'run selesai')

## 5. Bandingkan dengan FF @ LR utama dan BP-G

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
def per_run_of(res, keep):
    df = fp.load_results(res)
    df = df[~df.run.str.startswith(('tune_', 'cal_', 'pilot_'))]
    df = df[(df.width == WIDTH) & df.method.isin(keep)]
    pr, _ = fp.plasticity_table(df, window=10, last=10)
    return df, pr
df_main, pr_main = per_run_of(RES_MAIN, METHODS + ['bpg'])
df_s, pr_s = per_run_of(RES_S, METHODS)
pr_main['lr'] = [LR_MAIN[b][m] for b, m in zip(pr_main.bench, pr_main.method)]
pr_s['lr'] = LR_SENS
pr_s['method'] = pr_s['method'] + '@1e-3'
allr = pd.concat([pr_main, pr_s])
tab = allr.groupby(['bench', 'method', 'lr']).agg(
    n=('seed', 'count'), peak=('acc_peak', 'mean'), last=('acc_last', 'mean'),
    R_mean=('retention', 'mean'), R_sd=('retention', 'std'),
    dormant=('dormant_last', 'mean'), erank=('erank_last', 'mean')).reset_index()
rows = []
for bk in BENCH:
    for a, b in (('ff@1e-3', 'ff'), ('ff@1e-3', 'bpg')):
        xa = allr[(allr.bench == bk) & (allr.method == a)].retention
        xb = allr[(allr.bench == bk) & (allr.method == b)].retention
        if len(xa) > 1 and len(xb) > 1:
            rows.append(dict(bench=bk, A=a, B=b, mean_A=xa.mean(), mean_B=xb.mean(),
                             welch_p=stats.ttest_ind(xa, xb, equal_var=False).pvalue,
                             mwu_p=stats.mannwhitneyu(xa, xb, alternative='two-sided').pvalue,
                             n_A=len(xa), n_B=len(xb)))
tests = pd.DataFrame(rows)
tab.to_csv(f'{FIG}/lr_sensitivity.csv', index=False)
tests.to_csv(f'{FIG}/lr_sensitivity_tests.csv', index=False)
pd.set_option('display.width', 200)
display(tab.round(3)); display(tests.round(4))

## 6. Gambar A1: kurva akurasi FF pada dua LR (lampiran)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
MM = 1 / 25.4
plt.rcParams.update({'font.size': 7.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'pdf.fonttype': 42, 'savefig.dpi': 300})
series = [(df_main[df_main.method == 'ff'], 'FF, LR 3·10⁻³ (main)', '#D55E00', '-'),
          (df_s[df_s.method == 'ff'], 'FF, LR 10⁻³', '#D55E00', '--'),
          (df_main[df_main.method == 'bpg'], 'BP-G (main)', '#7B5EA7', '-')]
fig, axes = plt.subplots(1, 2, figsize=(190 * MM, 62 * MM), sharey=True)
for ax, (bk, title) in zip(axes, (('rlmnist', 'Random-label MNIST'), ('rlcifar', 'Random-label CIFAR-10'))):
    for d, lab, col, ls in series:
        g = d[d.bench == bk]
        if g.empty: continue
        piv = g.pivot_table(index='task', columns='seed', values='acc_end').rolling(3, min_periods=1).mean()
        mu, sd = piv.mean(1), piv.std(1).fillna(0)
        ax.plot(mu.index, mu, color=col, ls=ls, lw=1.3, label=lab)
        ax.fill_between(mu.index, mu - sd, mu + sd, color=col, alpha=0.15, lw=0)
    ax.axhline(0.1, color='0.5', lw=0.6, ls=':')
    ax.set_title(title); ax.set_xlabel('Task number'); ax.set_ylim(0, 1.04); ax.grid(alpha=0.25, lw=0.4)
axes[0].set_ylabel('Accuracy at end of task')
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.06))
fig.tight_layout(w_pad=1.2)
for ext in ('pdf', 'png'):
    fig.savefig(f'{FIG}/paper/Figure_A1.{ext}', bbox_inches='tight', pad_inches=0.02)
plt.close(fig); print('Figure_A1 tersimpan')

## 7. ZIP hasil (disimpan di Drive + diunduh)

In [ ]:
import zipfile, datetime
assert_drive()
EXP = f'{BASE}/exports'; os.makedirs(EXP, exist_ok=True)
ZIP_PATH = f'{EXP}/ff_lr_sensitivity_{datetime.datetime.now():%Y%m%d_%H%M}.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for dp, _, fs in os.walk(RES_S):
        for f in fs:
            if f.endswith(('.jsonl', '.done')):
                z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), BASE))
    for f in ('lr_sensitivity.csv', 'lr_sensitivity_tests.csv', 'paper/Figure_A1.pdf', 'paper/Figure_A1.png'):
        p = f'{FIG}/{f}'
        if os.path.exists(p): z.write(p, os.path.relpath(p, BASE))
os.sync()
print('ZIP:', ZIP_PATH, f'{os.path.getsize(ZIP_PATH) / 1e6:.1f} MB')
try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as e:
    print('Unduhan otomatis tidak tersedia; ambil dari Drive. (' + repr(e) + ')')